# Generación de imágenes de producto para E-commerce con Stable Diffusion

En este notebook se utiliza la librería `diffusers` de HuggingFace para generar imágenes de productos para una tienda en línea (e-commerce).

Se cubren los siguientes puntos:

1. Configuración del entorno.
2. Definición del modelo, scheduler y pipeline.
3. Generación de imágenes de producto (zapatillas deportivas) a partir de un prompt.
4. Experimentación con distintos prompts (bolso de lujo).
5. Generación de variaciones a partir de una imagen base (image-to-image).

## 1. Preparar el entorno

Instalación de las librerías necesarias.

In [ ]:
!pip install --upgrade diffusers torch transformers accelerate pillow

In [ ]:
import os
import torch
from PIL import Image

import warnings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo utilizado: {DEVICE}")

Definimos el directorio de salida donde se guardarán todas las imágenes generadas.

In [ ]:
OUTPUT_DIR = "ecommerce_generated_images/"

if not os.path.isdir(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

## 2. Definir el modelo y parámetros

Usamos el modelo `stabilityai/stable-diffusion-2-1-base` con el scheduler `EulerAncestralDiscreteScheduler`.

In [ ]:
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler

MODEL_ID = "stabilityai/stable-diffusion-2-1-base"

def create_pipeline(model_id=MODEL_ID):
    scheduler = EulerAncestralDiscreteScheduler.from_pretrained(
        model_id,
        subfolder="scheduler",
    )

    torch_dtype = torch.float16 if DEVICE == "cuda" else torch.float32

    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        scheduler=scheduler,
        torch_dtype=torch_dtype,
    ).to(DEVICE)

    return pipe

In [ ]:
pipe = create_pipeline()

## 3. Generación de imágenes de producto

Producto elegido: **zapatillas deportivas**.

Escribimos un prompt inicial describiendo el producto y ajustamos `guidance_scale` y `num_inference_steps` para obtener un resultado de alta calidad.

In [ ]:
sneakers_prompt = "A high-resolution product photo of modern running sneakers on a white background, studio lighting, commercial photography, sharp focus"
sneakers_negative_prompt = "blurry, low quality, deformed, watermark, text, cropped, out of frame"

generator = torch.Generator(device=DEVICE).manual_seed(42)

sneakers_image = pipe(
    sneakers_prompt,
    negative_prompt=sneakers_negative_prompt,
    guidance_scale=7.5,
    num_inference_steps=50,
    height=512,
    width=512,
    generator=generator,
).images[0]

sneakers_path = os.path.join(OUTPUT_DIR, "sneakers_base.png")
sneakers_image.save(sneakers_path)

sneakers_image

Probamos con otros valores de `guidance_scale` y `num_inference_steps` para comparar la calidad del resultado.

In [ ]:
generator = torch.Generator(device=DEVICE).manual_seed(42)

sneakers_image_v2 = pipe(
    sneakers_prompt,
    negative_prompt=sneakers_negative_prompt,
    guidance_scale=12,
    num_inference_steps=80,
    height=512,
    width=512,
    generator=generator,
).images[0]

sneakers_v2_path = os.path.join(OUTPUT_DIR, "sneakers_high_guidance.png")
sneakers_image_v2.save(sneakers_v2_path)

sneakers_image_v2

## 4. Experimentación con prompts

Probamos un producto distinto: un **bolso de lujo**, para comprobar la versatilidad del pipeline con otros prompts.

In [ ]:
handbag_prompt = "A luxury leather handbag with gold accents, studio lighting, white background, commercial product photography, high detail"
handbag_negative_prompt = "blurry, low quality, deformed, watermark, text, cropped, out of frame"

generator = torch.Generator(device=DEVICE).manual_seed(123)

handbag_image = pipe(
    handbag_prompt,
    negative_prompt=handbag_negative_prompt,
    guidance_scale=8.5,
    num_inference_steps=60,
    height=512,
    width=512,
    generator=generator,
).images[0]

handbag_path = os.path.join(OUTPUT_DIR, "handbag_gold_accents.png")
handbag_image.save(handbag_path)

handbag_image

Comparamos ambos productos generados en una misma cuadrícula.

In [ ]:
from diffusers.utils import make_image_grid

products_grid = make_image_grid([sneakers_image, sneakers_image_v2, handbag_image], rows=1, cols=3)
products_grid_path = os.path.join(OUTPUT_DIR, "products_comparison_grid.png")
products_grid.save(products_grid_path)

products_grid

## 5. Generación de imágenes a partir de otras imágenes (Image-to-Image)

Usamos la imagen base de las zapatillas generada anteriormente y aplicamos variaciones guiadas por texto: cambio de color y adición de un patrón.

In [ ]:
from diffusers import AutoPipelineForImage2Image

img2img_pipe = AutoPipelineForImage2Image.from_pipe(pipe)

In [ ]:
variation_prompt_1 = "Modern running sneakers in bright red color, white background, studio lighting, commercial photography"

generator = torch.Generator(device=DEVICE).manual_seed(7)

sneakers_variation_red = img2img_pipe(
    prompt=variation_prompt_1,
    image=sneakers_image,
    strength=0.6,
    guidance_scale=7.5,
    num_inference_steps=50,
    generator=generator,
).images[0]

variation_red_path = os.path.join(OUTPUT_DIR, "sneakers_variation_red.png")
sneakers_variation_red.save(variation_red_path)

sneakers_variation_red

In [ ]:
variation_prompt_2 = "Modern running sneakers with a colorful geometric pattern, white background, studio lighting, commercial photography"

generator = torch.Generator(device=DEVICE).manual_seed(21)

sneakers_variation_pattern = img2img_pipe(
    prompt=variation_prompt_2,
    image=sneakers_image,
    strength=0.7,
    guidance_scale=8,
    num_inference_steps=50,
    generator=generator,
).images[0]

variation_pattern_path = os.path.join(OUTPUT_DIR, "sneakers_variation_pattern.png")
sneakers_variation_pattern.save(variation_pattern_path)

sneakers_variation_pattern

Comparamos la imagen original de las zapatillas junto a sus dos variaciones (color rojo y patrón geométrico).

In [ ]:
variations_grid = make_image_grid(
    [sneakers_image, sneakers_variation_red, sneakers_variation_pattern],
    rows=1,
    cols=3,
)

variations_grid_path = os.path.join(OUTPUT_DIR, "sneakers_variations_grid.png")
variations_grid.save(variations_grid_path)

variations_grid

## Conclusiones

- Se configuró un pipeline de Stable Diffusion (`stable-diffusion-2-1-base`) con el scheduler `EulerAncestralDiscreteScheduler`.
- Se generaron imágenes de producto de alta calidad para dos productos ficticios de e-commerce (zapatillas y bolso), ajustando `guidance_scale` y `num_inference_steps`.
- Se aplicó la técnica image-to-image para generar variaciones del mismo producto (cambio de color y patrón) a partir de una imagen base.
- Todas las imágenes generadas se guardaron en el directorio `ecommerce_generated_images/`.